# TetraFT — QAFT for 2-bit Quaternary LLMs

Quantization-Aware Fine-Tuning on **Qwen2.5-0.5B** using the quaternary grid
{-1, -c, c, 1} with Straight-Through Estimator.

Runs on a single T4 (free Colab/Kaggle) in under 2 hours.

In [ ]:
# @title 1. Load TetraFT (from Kaggle dataset or local repo root)
import sys, os, json, math, logging

kaggle_path = '/kaggle/input/tetraft'
if os.path.isdir(kaggle_path):
    sys.path.insert(0, kaggle_path)
    print('Loaded from Kaggle dataset')
else:
    sys.path.insert(0, '.')
    print('Loaded from local repo root')

!pip install datasets --upgrade -q


In [ ]:
# @title 2. Imports
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from config import QAFTConfig
from model import replace_linear_layers
from train import QAFTTrainer
from eval import evaluate_perplexity

logging.basicConfig(level=logging.INFO, format='%(message)s')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"})')


In [ ]:
# @title 3. Configuration
cfg = QAFTConfig(
    model_name='Qwen/Qwen2.5-0.5B',
    quaternary_c=0.5,
    learning_rate=2e-5,
    batch_size=2,
    seq_length=512,
    max_steps=500,
    warmup_steps=50,
    gradient_accumulation_steps=4,
    logging_steps=10,
    eval_steps=100,
    save_steps=250,
    gradient_checkpointing=True,
    output_dir='./checkpoints',
)
print(cfg)


In [ ]:
# @title 4. Load Model & Tokenizer
print('Loading model...')
model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    torch_dtype=torch.float32,
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Model loaded to CPU (will move to GPU after layer replacement)')


In [ ]:
# @title 5. Replace Linear Layers with Quaternary
n_linear = sum(1 for _ in model.named_modules() if isinstance(_[1], nn.Linear))
print(f'Original nn.Linear layers: {n_linear}')

replace_linear_layers(model, c=cfg.quaternary_c, skip_lm_head=cfg.skip_lm_head)

n_quantized = sum(1 for _ in model.named_modules() if isinstance(_[1], type(model.layers[0].mlp.gate_proj)))
print(f'Replaced with QuantizedLinear: {n_quantized}')
model.to(device)
print(f'Model moved to {device}')


In [ ]:
# @title 6. Prepare Dataset (C4 sample)
print('Loading C4 sample (10,000 docs)...')
dataset = load_dataset(
    'c4', 'en',
    split='train',
    streaming=True,
    trust_remote_code=True,
)
sample = []
for i, doc in enumerate(dataset):
    if i >= 10_000:
        break
    sample.append(doc['text'])

def tokenize_fn(texts):
    return tokenizer(
        texts,
        truncation=True,
        padding='max_length',
        max_length=cfg.seq_length,
        return_tensors='pt',
    )

train_texts = sample[:9_000]
eval_texts = sample[9_000:]

def collate(batch):
    return tokenize_fn([b['text'] for b in batch])

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts):
        self.texts = texts
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        return {'text': self.texts[i]}

train_dataset = TextDataset(train_texts)
eval_dataset = TextDataset(eval_texts)

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate, num_workers=0)
eval_loader = DataLoader(eval_dataset, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate, num_workers=0)

print(f'Train batches: {len(train_loader)}, Eval batches: {len(eval_loader)}')


In [ ]:
# @title 7. Train
trainer = QAFTTrainer(model, tokenizer, cfg)
trainer.train(train_loader, eval_loader)


In [ ]:
# @title 8. Evaluate Final Perplexity
ppl = evaluate_perplexity(model, eval_loader, max_batches=20)
print(f'Final perplexity: {ppl:.2f}')


In [ ]:
# @title 9. Save & Load Checkpoint
# Checkpoint saved automatically during training in ./checkpoints/
# To load for continued training:
# from train import QAFTTrainer
# trainer = QAFTTrainer(model, tokenizer, cfg)
# trainer.load_checkpoint('/kaggle/input/checkpoint-dataset/checkpoint-best')


## Results

After training completes, record:
- **Pre-QAFT perplexity** (run eval before layer replacement as baseline)
- **Post-QAFT perplexity** (after training)
- **Perplexity delta** = post - pre (should be small, ideally < 2-3)

The quaternary compression saves **~87.5%** of linear layer parameter memory:
FP32 to 2-bit equivalent: 32 bits to 2 bits per weight.